In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

In [2]:
os.chdir("/Users/kaiping/Desktop/olist_project/data") 
os.getcwd()

'/Users/kaiping/Desktop/olist_project/data'

In [3]:
df_translation = pd.read_csv("raw/product_category_name_translation.csv")
df_sellers     = pd.read_csv("raw/olist_sellers_dataset.csv")
df_products    = pd.read_csv("raw/olist_products_dataset.csv")
df_orders      = pd.read_csv("raw/olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv("raw/olist_order_reviews_dataset.csv")
df_order_payments = pd.read_csv("raw/olist_order_payments_dataset.csv")
df_order_items    = pd.read_csv("raw/olist_order_items_dataset.csv")
df_geolocation    = pd.read_csv("raw/olist_geolocation_dataset.csv")
df_customers      = pd.read_csv("raw/olist_customers_dataset.csv")


### 客戶分群方式  
high: 貢獻總體收入前20%客人  
mid: 貢獻總體收入中間30%客人  
low: 貢獻總體收入後50%客人

In [4]:

# =========================================================
# 前提：
# 已先載入以下 DataFrame
# df_orders
# df_customers
# df_order_items
# df_order_reviews
# df_products
#
# 必要欄位：
# df_orders: [
#   "order_id", "customer_id", "order_status",
#   "order_purchase_timestamp",
#   "order_delivered_customer_date",
#   "order_estimated_delivery_date"
# ]
# df_customers: ["customer_id", "customer_unique_id"]
# df_order_items: ["order_id", "order_item_id", "product_id", "price", "freight_value"]
# df_order_reviews: ["order_id", "review_score"]
# df_products: ["product_id", "product_category_name"]
# =========================================================

# 1) 設定 analysis_end
analysis_end = pd.Timestamp("2018-05-31 15:00:00")

# 2) 日期欄位轉 datetime
orders = df_orders.copy()
for col in [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# 3) 篩選 delivered + 截斷 analysis_end 後資料
orders_base = (
    orders.loc[
        (orders["order_status"].eq("delivered")) &
        (orders["order_purchase_timestamp"] <= analysis_end),
        [
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .copy()
)

# 4) 併入 customer_unique_id
orders_base = orders_base.merge(
    df_customers[["customer_id", "customer_unique_id"]].drop_duplicates(),
    on="customer_id",
    how="left"
)

orders_base = orders_base.dropna(subset=["customer_unique_id"]).copy()

# 5) 建 order_items 明細，補 product category
order_items = df_order_items.copy()
order_items["price"] = pd.to_numeric(order_items["price"], errors="coerce")
order_items["freight_value"] = pd.to_numeric(order_items["freight_value"], errors="coerce")
order_items["item_gmv"] = order_items["price"].fillna(0) + order_items["freight_value"].fillna(0)

order_items = order_items.merge(
    df_products[["product_id", "product_category_name"]].drop_duplicates(),
    on="product_id",
    how="left"
)

# 6) 訂單層級 GMV / item 數
order_value = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        order_gmv=("item_gmv", "sum"),
        items_count=("order_item_id", "count")
    )
)

# 7) 訂單層級品類數（每單有幾種 category，可選保留；此處僅供中繼）
order_category_cnt = (
    order_items.groupby("order_id", as_index=False)
    .agg(order_unique_categories=("product_category_name", "nunique"))
)

# 8) review 先聚成 order 層級（避免一單多 review 造成重複）
order_review = (
    df_order_reviews.copy()
    .assign(review_score=lambda x: pd.to_numeric(x["review_score"], errors="coerce"))
    .groupby("order_id", as_index=False)
    .agg(avg_review_score_order=("review_score", "mean"))
)

# 9) 建立 order-level base
order_level = (
    orders_base
    .merge(order_value, on="order_id", how="left")
    .merge(order_category_cnt, on="order_id", how="left")
    .merge(order_review, on="order_id", how="left")
)

# 10) 計算配送延遲
# 只計算實際晚於預估的天數；提前送達視為 0
delay_days = (
    order_level["order_delivered_customer_date"] -
    order_level["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

order_level["delivery_delay_days"] = delay_days.clip(lower=0)
order_level["is_late"] = (delay_days > 0).astype(int)

# 11) 顧客層級時間特徵（需要 order 粒度去算）
order_level = order_level.sort_values(
    ["customer_unique_id", "order_purchase_timestamp", "order_id"]
).reset_index(drop=True)

order_level["gap_days"] = (
    order_level.groupby("customer_unique_id")["order_purchase_timestamp"]
    .diff()
    .dt.total_seconds() / (24 * 60 * 60)
)

# 12) 顧客層級主聚合（order-level）
customer_abt_main = (
    order_level.groupby("customer_unique_id", as_index=False)
    .agg(
        recency_days=("order_purchase_timestamp", lambda s: int((analysis_end - s.max()).days)),
        frequency_orders=("order_id", "nunique"),
        monetary_total=("order_gmv", "sum"),
        avg_order_value=("order_gmv", "mean"),
        avg_items_per_order=("items_count", "mean"),
        avg_review_score=("avg_review_score_order", "mean"),
        low_review_rate=("avg_review_score_order", lambda s: float((s <= 2).mean()) if s.notna().any() else np.nan),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        late_delivery_rate=("is_late", "mean"),
        customer_lifetime_days=("order_purchase_timestamp", lambda s: int((s.max() - s.min()).days)),
        avg_gap_days=("gap_days", "mean")
    )
)

# 13) 顧客層級 unique_categories
# 注意：這要從 item-level 算，因為一張訂單可能多個商品/品類
customer_categories = (
    orders_base[["order_id", "customer_unique_id"]]
    .merge(
        order_items[["order_id", "product_category_name"]],
        on="order_id",
        how="left"
    )
    .groupby("customer_unique_id", as_index=False)
    .agg(unique_categories=("product_category_name", "nunique"))
)

# 14) 合併顧客主表
customer_abt = customer_abt_main.merge(
    customer_categories,
    on="customer_unique_id",
    how="left"
)

# 15) repurchase_flag
customer_abt["repurchase_flag"] = (customer_abt["frequency_orders"] > 1).astype(int)

# 16) 型別與缺值整理
customer_abt["unique_categories"] = customer_abt["unique_categories"].fillna(0).astype(int)
customer_abt["frequency_orders"] = customer_abt["frequency_orders"].fillna(0).astype(int)
customer_abt["recency_days"] = customer_abt["recency_days"].astype(int)
customer_abt["customer_lifetime_days"] = customer_abt["customer_lifetime_days"].astype(int)

# 17) value_group
# high = monetary_total 前20%
# low  = monetary_total 後50%
# 其餘 = mid（若只想保留 high/low，可再過濾）
q80 = customer_abt["monetary_total"].quantile(0.80)
q50 = customer_abt["monetary_total"].quantile(0.50)

customer_abt["value_group"] = np.select(
    [
        customer_abt["monetary_total"] >= q80,
        customer_abt["monetary_total"] <= q50
    ],
    [
        "high",
        "low"
    ],
    default="mid"
)

customer_abt["value_group"] = pd.Categorical(
    customer_abt["value_group"],
    categories=["low", "mid", "high"],
    ordered=True
)

# 18) 最終欄位順序
customer_abt = customer_abt[
    [
        "customer_unique_id",
        "value_group",
        "recency_days",
        "frequency_orders",
        "monetary_total",
        "avg_order_value",
        "unique_categories",
        "avg_items_per_order",
        "avg_review_score",
        "low_review_rate",
        "avg_delivery_delay_days",
        "late_delivery_rate",
        "customer_lifetime_days",
        "avg_gap_days",
        "repurchase_flag"
    ]
].copy()

# 19) 若你這階段只要 high vs low，可啟用下面這行
# customer_abt = customer_abt[customer_abt["value_group"].isin(["high", "low"])].copy()

print(customer_abt.shape)
display(customer_abt.head())

(75320, 15)


,customer_unique_id,value_group,recency_days,frequency_orders,monetary_total,avg_order_value,unique_categories,avg_items_per_order,avg_review_score,low_review_rate,avg_delivery_delay_days,late_delivery_rate,customer_lifetime_days,avg_gap_days,repurchase_flag
0,0000366f3b9a7992bf8c76cfdf3221e2,mid,21,1,141.90,141.90,1,1.0,5.0,0.0,0.0,0.0,0,NaN,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,low,24,1,27.19,27.19,1,1.0,4.0,0.0,0.0,0.0,0,NaN,0
2,0000f46a3911fa3c0805444483337064,low,446,1,86.22,86.22,1,1.0,3.0,0.0,0.0,0.0,0,NaN,0
3,0000f6ccb0745a6a4b88665a16c9f078,low,230,1,43.62,43.62,1,1.0,4.0,0.0,0.0,0.0,0,NaN,0
4,0004aac84e0df4da2b147fca70cf8255,mid,197,1,196.89,196.89,1,1.0,5.0,0.0,0.0,0.0,0,NaN,0


In [5]:
customer_abt.isna().sum()

customer_unique_id             0
value_group                    0
recency_days                   0
frequency_orders               0
monetary_total                 0
avg_order_value                0
unique_categories              0
avg_items_per_order            0
avg_review_score             524
low_review_rate              524
avg_delivery_delay_days        2
late_delivery_rate             0
customer_lifetime_days         0
avg_gap_days               73092
repurchase_flag                0
dtype: int64

### 缺失修補邏輯  
1. avg_gap_days: 保留不補
2. 新增欄位 has_review 和 has_gap
### 補值
1. avg_review_score：缺失補中位數
2. low_review_rate: 補0
3. avg_delivery_delay_days：補0

In [6]:
# =========================================================
# 缺失處理：EDA 版本
# 原則：
# 1. gap 缺失屬於結構性缺失（無回購），不直接硬補平均值
# 2. review 缺失代表沒有評論，保留訊號並補值
# 3. delivery delay 少量缺失補 0
# =========================================================

customer_abt_clean = customer_abt.copy()

# 1) review 缺失旗標
customer_abt_clean["has_review"] = customer_abt_clean["avg_review_score"].notna().astype(int)

# 2) gap 缺失旗標
customer_abt_clean["has_gap"] = customer_abt_clean["avg_gap_days"].notna().astype(int)

# 3) avg_review_score
# 用中位數補，避免平均數受偏態影響
review_median = customer_abt_clean["avg_review_score"].median()
customer_abt_clean["avg_review_score"] = customer_abt_clean["avg_review_score"].fillna(review_median)

# 4) low_review_rate
# 沒有 review 時，先補 0；因為沒有觀測到低分
customer_abt_clean["low_review_rate"] = customer_abt_clean["low_review_rate"].fillna(0)

# 5) avg_delivery_delay_days
# delivered 訂單少數日期缺失，先補 0
customer_abt_clean["avg_delivery_delay_days"] = customer_abt_clean["avg_delivery_delay_days"].fillna(0)

# 6) avg_gap_days
# EDA 階段保留原始 NaN，另建一個可分析/可建模版本
customer_abt_clean["avg_gap_days_filled"] = customer_abt_clean["avg_gap_days"].fillna(999)

# 7) 檢查
print(customer_abt_clean.isna().sum())

customer_unique_id             0
value_group                    0
recency_days                   0
frequency_orders               0
monetary_total                 0
avg_order_value                0
unique_categories              0
avg_items_per_order            0
avg_review_score               0
low_review_rate                0
avg_delivery_delay_days        0
late_delivery_rate             0
customer_lifetime_days         0
avg_gap_days               73092
repurchase_flag                0
has_review                     0
has_gap                        0
avg_gap_days_filled            0
dtype: int64


In [7]:
customer_abt_clean.shape

(75320, 18)

In [8]:
customer_abt_clean.columns

Index(['customer_unique_id', 'value_group', 'recency_days', 'frequency_orders',
       'monetary_total', 'avg_order_value', 'unique_categories',
       'avg_items_per_order', 'avg_review_score', 'low_review_rate',
       'avg_delivery_delay_days', 'late_delivery_rate',
       'customer_lifetime_days', 'avg_gap_days', 'repurchase_flag',
       'has_review', 'has_gap', 'avg_gap_days_filled'],
      dtype='object')

In [11]:
customer_abt_clean.dtypes

customer_unique_id           object
value_group                category
recency_days                  int64
frequency_orders              int64
monetary_total              float64
avg_order_value             float64
unique_categories             int64
avg_items_per_order         float64
avg_review_score            float64
low_review_rate             float64
avg_delivery_delay_days     float64
late_delivery_rate          float64
customer_lifetime_days        int64
avg_gap_days                float64
repurchase_flag               int64
has_review                    int64
has_gap                       int64
avg_gap_days_filled         float64
dtype: object

### 高價值顧客與購買節奏差異

In [18]:
# 0) 準備資料
# =========================================================
df = customer_abt_clean.copy()

# 只保留 high / low
df_hl = df[df["value_group"].isin(["high", "low"])].copy()

# 補回購次數
df_hl["repeat_order_count"] = (df_hl["frequency_orders"] - 1).clip(lower=0)

# 確保類別順序
df_hl["value_group"] = pd.Categorical(
    df_hl["value_group"],
    categories=["low", "high"],
    ordered=True
)


In [19]:
df_hl.shape

(52724, 19)

In [21]:
df_hl.isna().sum()

customer_unique_id             0
value_group                    0
recency_days                   0
frequency_orders               0
monetary_total                 0
avg_order_value                0
unique_categories              0
avg_items_per_order            0
avg_review_score               0
low_review_rate                0
avg_delivery_delay_days        0
late_delivery_rate             0
customer_lifetime_days         0
avg_gap_days               51253
repurchase_flag                0
has_review                     0
has_gap                        0
avg_gap_days_filled            0
repeat_order_count             0
dtype: int64

In [23]:
df_hl.loc[df_hl["value_group"] == "high"].describe()

,recency_days,frequency_orders,monetary_total,avg_order_value,unique_categories,avg_items_per_order,avg_review_score,low_review_rate,avg_delivery_delay_days,late_delivery_rate,customer_lifetime_days,avg_gap_days,repurchase_flag,has_review,has_gap,avg_gap_days_filled,repeat_order_count
count,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.00000,1206.000000,15064.000000,15064.000000,15064.000000,15064.000000,15064.000000
mean,194.450212,1.093401,439.215378,420.113472,1.055364,1.349717,4.010683,0.167620,1.095121,0.097881,6.32269,70.330020,0.080058,0.990839,0.080058,924.652151,0.093401
std,133.590505,0.357726,379.717192,373.419577,0.291943,0.922411,1.393507,0.369252,6.249400,0.292961,35.74114,92.862771,0.271393,0.095276,0.271393,253.399065,0.357726
min,0.000000,1.000000,208.080000,54.990000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,83.000000,1.000000,245.830000,236.475000,1.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.00000,0.011285,0.000000,1.000000,0.000000,999.000000,0.000000
50%,177.000000,1.000000,317.800000,306.005000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.00000,30.549306,0.000000,1.000000,0.000000,999.000000,0.000000
75%,290.000000,1.000000,466.167500,443.585000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.00000,108.821007,0.000000,1.000000,0.000000,999.000000,0.000000
max,604.000000,10.000000,13664.080000,13664.080000,5.000000,20.000000,5.000000,1.000000,181.608333,1.000000,572.00000,572.686806,1.000000,1.000000,1.000000,999.000000,9.000000


In [24]:
df_hl.loc[df_hl["value_group"] == "low"].describe()

,recency_days,frequency_orders,monetary_total,avg_order_value,unique_categories,avg_items_per_order,avg_review_score,low_review_rate,avg_delivery_delay_days,late_delivery_rate,customer_lifetime_days,avg_gap_days,repurchase_flag,has_review,has_gap,avg_gap_days_filled,repeat_order_count
count,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.00000,265.000000,37660.000000,37660.000000,37660.000000,37660.000000,37660.000000
mean,196.746123,1.007223,63.371930,63.081659,0.990414,1.048593,4.179056,0.117910,0.781221,0.080072,0.37419,53.152728,0.007037,0.993893,0.007037,992.344410,0.007223
std,131.731519,0.087152,23.311317,23.301645,0.139375,0.243046,1.247324,0.322269,4.670067,0.271200,8.21462,82.405860,0.083590,0.077911,0.083590,79.364008,0.087152
min,0.000000,1.000000,10.070000,10.070000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,88.000000,1.000000,44.000000,43.690000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000,0.000000,999.000000,0.000000
50%,178.000000,1.000000,63.000000,62.540000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.00000,11.649306,0.000000,1.000000,0.000000,999.000000,0.000000
75%,294.250000,1.000000,82.330000,81.970000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.00000,74.679167,0.000000,1.000000,0.000000,999.000000,0.000000
max,604.000000,4.000000,107.200000,107.200000,3.000000,6.000000,5.000000,1.000000,167.708333,1.000000,358.00000,358.941667,1.000000,1.000000,1.000000,999.000000,3.000000


In [25]:
# RFM對比分析
rfm_compare = (
    customer_abt_clean
    .groupby("value_group", observed=True)[
        ["recency_days", "frequency_orders", "monetary_total"]
    ]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

rfm_compare

recency_days                        frequency_orders               \
                   count    mean median     std            count  mean median   
value_group                                                                     
low                37660  196.75  178.0  131.73            37660  1.01    1.0   
mid                22596  190.21  172.0  131.21            22596  1.04    1.0   
high               15064  194.45  177.0  133.59            15064  1.09    1.0   

                  monetary_total                          
              std          count    mean  median     std  
value_group                                               
low          0.09          37660   63.37   63.00   23.31  
mid          0.20          22596  149.33  146.12   28.03  
high         0.36          15064  439.22  317.80  379.72

In [26]:
# 平均客單價分析
aov_summary = (
    customer_abt_clean
    .groupby("value_group", observed=True)["avg_order_value"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

aov_summary

,count,mean,median,std,min,max
value_group,,,,,,
low,37660,63.08,62.54,23.30,10.07,107.20
mid,22596,146.64,144.11,30.48,18.45,208.07
high,15064,420.11,306.00,373.42,54.99,13664.08


In [27]:
aov_quantiles = (
    customer_abt_clean
    .groupby("value_group", observed=True)["avg_order_value"]
    .quantile([0.25, 0.5, 0.75, 0.9, 0.95])
    .unstack()
    .round(2)
)

aov_quantiles

,0.25,0.50,0.75,0.90,0.95
value_group,,,,,
low,43.69,62.54,81.97,96.82,102.09
mid,122.56,144.11,169.23,190.37,198.29
high,236.48,306.00,443.58,756.49,1054.51


In [28]:
aov_compare = (
    customer_abt_clean
    .groupby("value_group", observed=True)
    .agg(
        customer_count=("customer_unique_id", "size"),
        aov_mean=("avg_order_value", "mean"),
        aov_median=("avg_order_value", "median"),
        aov_std=("avg_order_value", "std"),
    )
    .round(2)
)

aov_compare

,customer_count,aov_mean,aov_median,aov_std
value_group,,,,
low,37660,63.08,62.54,23.30
mid,22596,146.64,144.11,30.48
high,15064,420.11,306.00,373.42


In [29]:
high = aov_compare.loc["high"]
low = aov_compare.loc["low"]

aov_diff = pd.DataFrame({
    "metric": aov_compare.columns,
    "high": high.values,
    "low": low.values
})

aov_diff["diff_high_minus_low"] = aov_diff["high"] - aov_diff["low"]
aov_diff["pct_diff_vs_low"] = (
    aov_diff["diff_high_minus_low"] / aov_diff["low"].replace(0, pd.NA)
) * 100

aov_diff.round(2)

,metric,high,low,diff_high_minus_low,pct_diff_vs_low
0,customer_count,15064.00,37660.00,-22596.00,-60.00
1,aov_mean,420.11,63.08,357.03,566.00
2,aov_median,306.00,62.54,243.46,389.29
3,aov_std,373.42,23.30,350.12,1502.66
